# GPT-2 Token-Level Perplexity & Burstiness Feature Extraction (v2)

This standalone Google Colab notebook computes token-level autoregressive language model features using **GPT-2** (`gpt2` small) on GPU:
1. **`mean_log_prob`**: Mean per-token log-probability under GPT-2.
2. **`perplexity`**: Exponentiated negative mean log-prob ($\exp(-\text{mean\_log\_prob})$).
3. **`std_log_prob`**: Standard deviation of per-token log-probs (representing burstiness / variability).
4. **`token_count`**: Number of scored transition tokens (excluding padding).

### Inputs & Outputs
- **Inputs**: `unified_dataset.csv` and `synthetic_hotel_ai.csv` (from Google Drive or direct `/content/` upload).
- **Cleaning**: Exact standalone implementation of `clean_text()` (HTML unescaping, `<br>` tag removal, unicode quotes/dashes normalization, chat preamble stripping).
- **Output**: `gpt2_perplexity_v2.csv` containing `(source_file, row_id, token_count, mean_log_prob, perplexity, std_log_prob)`.

In [ ]:
# Cell 1: Environment & GPU Setup
!nvidia-smi
!pip install --quiet transformers torch pandas numpy tqdm

In [ ]:
# Cell 2: Imports & Device Configuration
import os
import html
import re
import math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Cell 3: Standalone clean_text() Implementation (Exact match to ml/src/data/cleaning.py)
PREAMBLE_PATTERN = re.compile(
    r"^(?:(?:Sure|Certainly|Of course)[!,.]?\s*(?:Here(?:\'s| is)|I(?:\'d| would) be happy to|below is|this is)?|"
    r"(?:Here(?:\'s| is)|Below is)\s+(?:a|an|the|my)?\s*(?:AI|sample|generated|draft|possible)?\s*(?:IMDb\s+)?review\b)"
    r"[^:\n]{0,150}:\s*",
    re.IGNORECASE,
)

HTML_TAG_PATTERN = re.compile(r"<[^>]+>")

UNICODE_QUOTES_DASHES = {
    "\u2018": "'",
    "\u2019": "'",
    "\u201a": "'",
    "\u201b": "'",
    "\u201c": '"',
    "\u201d": '"',
    "\u201e": '"',
    "\u201f": '"',
    "\u00ab": '"',
    "\u00bb": '"',
    "\u2014": " - ",
    "\u2013": " - ",
    "\u2015": " - ",
    "\u2212": "-",
    "\u00a0": " ",
    "\u2026": "...",
    "`": "'",
    "´": "'",
}

def clean_text(text: str) -> str:
    """
    Standard text cleaning pipeline for review text:
    1. Unescapes HTML entities (&amp;, &quot;, &#39;, etc.)
    2. Strips HTML tags (<br />, <p>, etc.)
    3. Normalizes unicode quotes, apostrophes, dashes, and spaces
    4. Removes leading chat preambles (e.g. 'Sure! Here\'s my review of ...:')
    5. Collapses multiple whitespace characters to a single space and strips.
    """
    if not isinstance(text, str) or not text:
        return ""

    cleaned = html.unescape(text)
    cleaned = HTML_TAG_PATTERN.sub(" ", cleaned)

    for char, replacement in UNICODE_QUOTES_DASHES.items():
        if char in cleaned:
            cleaned = cleaned.replace(char, replacement)

    cleaned = PREAMBLE_PATTERN.sub("", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

# Verification test on sample artifact strings
test_str = "Sure! Here\'s my review of the film: \"Toy Story 3\" was great.<br /><br />Loved it!"
print("Sample Cleaning Test:")
print("  Before:", repr(test_str))
print("  After: ", repr(clean_text(test_str)))

In [ ]:
# Cell 4: Locate & Load Datasets (Support Google Drive or direct /content/ upload)
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception as e:
    print(f"Google Drive mount note: {e}")

def locate_file(filename: str) -> Path:
    candidates = [
        Path(f"/content/{filename}"),
        Path(f"/content/drive/MyDrive/fake-review-detector/{filename}"),
        Path(f"/content/drive/MyDrive/fake-review-detector/data/processed/{filename}"),
        Path(f"data/processed/{filename}"),
        Path(filename),
    ]
    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError(
        f"Could not find '{filename}'.\n"
        f"Please upload '{filename}' directly to Colab via the Files pane or mount Google Drive."
    )

unified_path = locate_file("unified_dataset.csv")
synth_path = locate_file("synthetic_hotel_ai.csv")

print(f"Loading unified_dataset from: {unified_path}")
unified_df = pd.read_csv(unified_path)
print(f"  Loaded {len(unified_df):,} rows. Columns: {list(unified_df.columns)}")

print(f"Loading synthetic_hotel_ai from: {synth_path}")
synth_df = pd.read_csv(synth_path)
print(f"  Loaded {len(synth_df):,} rows. Columns: {list(synth_df.columns)}")

# Verify required columns exist
assert "row_id" in unified_df.columns and "text" in unified_df.columns, "unified_dataset.csv missing 'row_id' or 'text'"
assert "row_id" in synth_df.columns and "text" in synth_df.columns, "synthetic_hotel_ai.csv missing 'row_id' or 'text'"
print("\nVerified: Both files contain 'row_id' and 'text' columns cleanly.")

In [ ]:
# Cell 5: Apply clean_text() to every row BEFORE tokenizing
print("Applying clean_text() to unified_dataset.csv...")
unified_df["text_clean"] = unified_df["text"].apply(clean_text)

print("Applying clean_text() to synthetic_hotel_ai.csv...")
synth_df["text_clean"] = synth_df["text"].apply(clean_text)

print("Cleaning complete. Sample cleaned row:")
print("Row ID:", unified_df.iloc[0]["row_id"])
print("Text:  ", repr(unified_df.iloc[0]["text_clean"][:150]))

In [ ]:
# Cell 6: Load GPT-2 Model & Tokenizer on CUDA (fp16)
model_id = "gpt2"
print(f"Loading {model_id} tokenizer and model...")

tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = GPT2LMHeadModel.from_pretrained(model_id, torch_dtype=dtype).to(device)
model.eval()

print(f"Model loaded on {device} with dtype: {dtype}")
print(f"Tokenizer pad_token set to: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"Padding side: {tokenizer.padding_side}")

In [ ]:
# Cell 7: Batch Computation of Per-Token Log-Probs, Perplexity & Burstiness
def compute_gpt2_features(df: pd.DataFrame, text_col: str = "text_clean", batch_size: int = 32, max_length: int = 256) -> pd.DataFrame:
    """
    Computes per-row features in batches (batch_size=32, max_length=256, truncation=True) using torch.no_grad():
    - Shifts logits[:, :-1, :] and input_ids[:, 1:] to align predictions with targets.
    - Calculates per-token log-probs ONLY over non-padding tokens using shifted attention_mask.
    - Computes mean_log_prob, perplexity (exp(-mean_log_prob)), std_log_prob, and token_count.
    - Safely handles edge cases (< 2 tokens) by setting metrics to NaN.
    """
    texts = df[text_col].tolist()
    row_ids = df["row_id"].tolist()

    mean_log_probs = []
    perplexities = []
    std_log_probs = []
    token_counts = []

    short_row_indices = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Computing GPT-2 Features"):
        batch_texts = texts[i : i + batch_size]

        # Tokenize with right padding and truncation to max_length=256
        encodings = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        input_ids = encodings["input_ids"].to(device)
        attention_mask = encodings["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model(input_ids)
            logits = outputs.logits  # Shape: (B, T, vocab_size)

            # Shift logits, labels, and attention mask
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = input_ids[:, 1:].contiguous()
            shift_mask = attention_mask[:, 1:].contiguous()

            # Log-softmax over vocabulary
            log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
            # Gather log-probabilities for true target tokens
            target_log_probs = log_probs.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)

            for b in range(len(batch_texts)):
                global_idx = i + b
                row_mask = shift_mask[b].bool()
                valid_log_probs = target_log_probs[b][row_mask]
                num_tokens = valid_log_probs.numel()

                # Handle < 2 tokens (i.e. < 1 scored transition token)
                if num_tokens < 1:
                    short_row_indices.append((global_idx, row_ids[global_idx], batch_texts[b]))
                    mean_log_probs.append(np.nan)
                    perplexities.append(np.nan)
                    std_log_probs.append(np.nan)
                    token_counts.append(0)
                else:
                    lp_np = valid_log_probs.float().cpu().numpy()
                    m_lp = float(lp_np.mean())
                    s_lp = float(lp_np.std()) if num_tokens > 1 else 0.0
                    try:
                        ppl = float(np.exp(-m_lp))
                    except OverflowError:
                        ppl = float("inf")

                    mean_log_probs.append(m_lp)
                    perplexities.append(ppl)
                    std_log_probs.append(s_lp)
                    token_counts.append(num_tokens)

    if short_row_indices:
        print(f"Warning: {len(short_row_indices)} rows had < 2 tokens and were assigned NaN metrics.")
        for s_idx, s_id, s_txt in short_row_indices[:5]:
            print(f"  [short] idx={s_idx}, row_id={s_id}, text={repr(s_txt[:50])}")
    else:
        print("All rows had >= 2 tokens. Zero short rows detected.")

    return pd.DataFrame({
        "row_id": row_ids,
        "token_count": token_counts,
        "mean_log_prob": mean_log_probs,
        "perplexity": perplexities,
        "std_log_prob": std_log_probs,
    })

# Run feature extraction for unified_dataset
print("\n--- Computing features for unified_dataset.csv ---")
unified_features = compute_gpt2_features(unified_df, text_col="text_clean", batch_size=32, max_length=256)
unified_features["source_file"] = "unified_dataset"

# Run feature extraction for synthetic_hotel_ai
print("\n--- Computing features for synthetic_hotel_ai.csv ---")
synth_features = compute_gpt2_features(synth_df, text_col="text_clean", batch_size=32, max_length=256)
synth_features["source_file"] = "synthetic_hotel_ai"

In [ ]:
# Cell 8: Concatenate, Assert Zero Duplicates, & Export to gpt2_perplexity_v2.csv
out_df = pd.concat([unified_features, synth_features], ignore_index=True)

# Order columns cleanly
out_df = out_df[["source_file", "row_id", "token_count", "mean_log_prob", "perplexity", "std_log_prob"]]

# Assert zero duplicate (source_file, row_id) pairs
dup_count = out_df.duplicated(subset=["source_file", "row_id"]).sum()
assert dup_count == 0, f"Duplicate check failed: found {dup_count} duplicate (source_file, row_id) pairs!"
print(f"Verified: Exactly 0 duplicate (source_file, row_id) pairs found across {len(out_df):,} total rows.")

output_filename = "gpt2_perplexity_v2.csv"
out_df.to_csv(output_filename, index=False)
print(f"Successfully saved output to '{output_filename}' ({Path(output_filename).stat().st_size / 1e6:.2f} MB).")

In [ ]:
# Cell 9: Print Summary Statistics (row counts, NaN counts, mean/median perplexity by source_file)
print("=" * 80)
print("GPT-2 PERPLEXITY & BURSTINESS SUMMARY STATISTICS")
print("=" * 80)

for src, grp in out_df.groupby("source_file"):
    total = len(grp)
    nan_ppl = grp["perplexity"].isna().sum()
    valid_ppl = grp["perplexity"].dropna()
    valid_mlp = grp["mean_log_prob"].dropna()
    valid_slp = grp["std_log_prob"].dropna()
    
    print(f"\nSource File: [{src}]")
    print(f"  Total Rows:           {total:,}")
    print(f"  Rows with NaN PPL:    {nan_ppl} ({nan_ppl / total * 100:.2f}%)")
    print(f"  Mean Perplexity:      {valid_ppl.mean():.4f}")
    print(f"  Median Perplexity:    {valid_ppl.median():.4f}")
    print(f"  Std Perplexity:       {valid_ppl.std():.4f}")
    print(f"  Mean Log-Probability: {valid_mlp.mean():.4f}")
    print(f"  Mean Burstiness (Std):{valid_slp.mean():.4f}")
    print(f"  Avg Tokens Scored:    {grp['token_count'].mean():.2f}")

print("\nFirst 5 Rows of Export:")
print(out_df.head(5).to_string(index=False))
print("=" * 80)

In [ ]:
# Cell 10: Download / Export File from Colab
try:
    from google.colab import files
    files.download("gpt2_perplexity_v2.csv")
    print("Browser download initiated for 'gpt2_perplexity_v2.csv'.")
except Exception as e:
    print(f"Direct browser download not triggered: {e}")

# If Drive is mounted, copy to Google Drive project folder
drive_target_dir = Path("/content/drive/MyDrive/fake-review-detector/data/processed")
if not drive_target_dir.exists():
    drive_target_dir = Path("/content/drive/MyDrive/fake-review-detector")

if drive_target_dir.exists():
    import shutil
    dest_path = drive_target_dir / "gpt2_perplexity_v2.csv"
    shutil.copy("gpt2_perplexity_v2.csv", dest_path)
    print(f"Copied output file to Google Drive: {dest_path}")